In [ ]:
import os
from experiments.common import paths


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # For debugging CUDA errors

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import torchvision
import os
from tqdm.auto import tqdm
from fff.evaluate.fid import compute_fid_openai_tf as compute_fid
from fff.evaluate.fid_old import compute_fid as compute_fid_alternative
import matplotlib as mpl

mpl.rcParams["mathtext.fontset"] = "stix"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
@torch.no_grad()
def find_nearest_neighbor(image_embeddings, original_embeddings, identity_threshold=1.e-4):
    distance_matrix = torch.mean((image_embeddings[None, ...] - original_embeddings[:, None, ...]) ** 2, dim=(-1))
    distance_matrix[distance_matrix < identity_threshold] = torch.inf
    return torch.argmin(distance_matrix, dim=0)

In [ ]:
@torch.no_grad()
def find_nearest_neighbor_batched_mm(
    image_embeddings: torch.Tensor,
    dataset: torch.Tensor,
    subject_model: torch.nn.Module,
    identity_threshold: float = 1e-4,
    batch_size: int = 1024,
    metric="l2",
):
    device = image_embeddings.device
    N_img, D = image_embeddings.shape
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)

    best_dist = torch.full((N_img,), torch.inf, device=device)
    best_idx = torch.full((N_img,), -1, dtype=torch.long, device=device)

    if metric == "l2":
        img_norms = (image_embeddings ** 2).mean(dim=1)  # (N_img,)

    elif metric == "cross_entropy":
        # treat image_embeddings as target distributions
        img_probs = torch.softmax(image_embeddings, dim=-1)

    for n_batch, batch in enumerate(tqdm(dataloader)):
        orig_chunk = subject_model(batch[0].to(device))  # (B, D)
        B = orig_chunk.shape[0]

        if metric == "l2":
            orig_norms = (orig_chunk ** 2).mean(dim=1)  # (B,)
            cross = orig_chunk @ image_embeddings.T     # (B, N_img)

            dist = (
                orig_norms[:, None]
                + img_norms[None, :]
                - 2 * cross / D
            )

        elif metric == "l1":
            # (B, N_img, D) → sum over D
            dist = (orig_chunk[:, None, :] - image_embeddings[None, :, :]).abs().mean(dim=-1)

        elif metric == "cross_entropy":
            # CE(p_img || q_orig) = - sum p_img * log q_orig
            log_q = torch.log_softmax(orig_chunk, dim=-1)  # (B, D)

            # (B, N_img)
            dist = -(img_probs[None, :, :] * log_q[:, None, :]).sum(dim=-1)

        else:
            raise ValueError(f"Unknown metric: {metric}")

        # Ignore identical embeddings
        dist[dist < identity_threshold] = torch.inf

        # Best match in this chunk
        chunk_min_dist, chunk_min_idx = torch.min(dist, dim=0)

        # Update global best
        update = chunk_min_dist < best_dist
        best_dist[update] = chunk_min_dist[update]
        best_idx[update] = chunk_min_idx[update] + n_batch * batch_size

    return best_idx

In [ ]:
def masked_mean(x, mask, dim):
    mask = mask.float().to(x.device)
    return (x * mask).sum(dim) / mask.sum(dim).clamp(min=1)

def masked_std(x, mask, dim, eps=1e-8):
    mask = mask.float().to(x.device)
    
    mean = (x * mask).sum(dim, keepdim=True) / mask.sum(dim, keepdim=True).clamp(min=1)
    var = ((x - mean) ** 2 * mask).sum(dim) / mask.sum(dim).clamp(min=1)
    
    return (var + eps).sqrt()

In [ ]:
def combine_and_save_imagenet_samples(prefix_filter="sampled_imagenet_invariances_", save_name="sampled_imagenet_invariances.pt"):
    combined_dict = {
        "invariances": [],
        "originals": [],
        "labels": [],
        "invariances_embeddings": [],
        "original_embeddings": [],
    }
    
    for foldername in tqdm(os.listdir(".")):
        if foldername.startswith(prefix_filter):
            results_dict = {
                            "invariances": [],
                            "originals": [],
                            "labels": [],
                            "invariances_embeddings": [],
                            "original_embeddings": [],
                           }
            chunk_id = 0
            while os.path.exists(os.path.join(foldername, f"chunk_{chunk_id}.pt")):
                chunk_dict = torch.load(os.path.join(foldername, f"chunk_{chunk_id}.pt"))
                for key in results_dict.keys():
                    results_dict[key].append(chunk_dict[key])
                chunk_id += 1
            for key in results_dict.keys():
                combined_dict[key].append(torch.cat(results_dict[key], dim=0))
    
    # Check consistency:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        assert torch.all(((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=-1) < 1.e-4), "Dataset order does not agree"
    
    max_len, ind_max = 0, None
    for i, orig in enumerate(combined_dict["originals"]):
        if len(orig) > max_len:
            ind_max = i
            max_len = len(orig)
    
    combined_dict["originals"] = combined_dict["originals"][ind_max]
    combined_dict["original_embeddings"] = combined_dict["original_embeddings"][ind_max]
    combined_dict["labels"] = combined_dict["labels"][ind_max]
    
    assert len(combined_dict["originals"]) == len(combined_dict["original_embeddings"]) == len(combined_dict["labels"])
    
    combined_dict["masks"] = []
    for i, (inv, inv_embeddings) in enumerate(zip(combined_dict["invariances"], combined_dict["invariances_embeddings"])):
        assert len(inv) == len(inv_embeddings)
        mask = torch.cat((torch.ones(len(inv), device=inv.device, dtype=bool), 
                          torch.zeros(max_len - len(inv), device=inv.device, dtype=bool)), dim=0)
        inv = torch.cat((inv, torch.zeros(max_len - len(inv), *inv.shape[1:], device=inv.device, dtype=inv.dtype)), dim=0)
        inv_embeddings = torch.cat((inv_embeddings, torch.zeros(max_len - len(inv_embeddings), *inv_embeddings.shape[1:], device=inv_embeddings.device, dtype=inv_embeddings.dtype)), dim=0)
        combined_dict["invariances"][i] = inv
        combined_dict["invariances_embeddings"][i] = inv_embeddings
        combined_dict["masks"].append(mask)
    
    combined_dict["invariances"] = torch.stack(combined_dict["invariances"], dim=1)
    combined_dict["invariances_embeddings"] = torch.stack(combined_dict["invariances_embeddings"], dim=1)
    combined_dict["masks"] = torch.stack(combined_dict["masks"], dim=1)
    
    torch.save(combined_dict, save_name)

# DINO augmentations

In [ ]:
# Taken from https://github.com/facebookresearch/dinov2/blob/main/dinov2/data/augmentations.py

import logging

from torchvision import transforms
from typing import Sequence
from torch import nn


IMAGENET_DEFAULT_MEAN = (0.5, 0.5, 0.5)
IMAGENET_DEFAULT_STD = (0.5, 0.5, 0.5)

class GaussianBlur(transforms.RandomApply):
    """
    Apply Gaussian Blur to the PIL image.
    """

    def __init__(self, *, p: float = 0.5, radius_min: float = 0.1, radius_max: float = 2.0):
        # NOTE: torchvision is applying 1 - probability to return the original image
        keep_p = 1 - p
        transform = transforms.GaussianBlur(kernel_size=9, sigma=(radius_min, radius_max))
        super().__init__(transforms=[transform], p=keep_p)

def make_normalize_transform(
    mean: Sequence[float] = IMAGENET_DEFAULT_MEAN,
    std: Sequence[float] = IMAGENET_DEFAULT_STD,
) -> transforms.Normalize:
    return transforms.Normalize(mean=mean, std=std)

class DataAugmentationDINO(object):
    def __init__(
        self,
        global_crops_scale,
        local_crops_scale,
        local_crops_number,
        global_crops_size=224,
        local_crops_size=96,
    ):
        self.global_crops_scale = global_crops_scale
        self.local_crops_scale = local_crops_scale
        self.local_crops_number = local_crops_number
        self.global_crops_size = global_crops_size
        self.local_crops_size = local_crops_size

        # random resized crop and flip
        self.geometric_augmentation_global = transforms.Compose(
            [
                transforms.RandomResizedCrop(
                    global_crops_size, scale=global_crops_scale, interpolation=transforms.InterpolationMode.BICUBIC
                ),
                transforms.RandomHorizontalFlip(p=0.5),
            ]
        )

        self.geometric_augmentation_local = transforms.Compose(
            [
                transforms.RandomResizedCrop(
                    local_crops_size, scale=local_crops_scale, interpolation=transforms.InterpolationMode.BICUBIC
                ),
                transforms.RandomHorizontalFlip(p=0.5),
            ]
        )

        # color distorsions / blurring
        color_jittering = transforms.Compose(
            [
                transforms.RandomApply(
                    [transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)],
                    p=0.8,
                ),
                transforms.RandomGrayscale(p=0.2),
            ]
        )

        global_transfo1_extra = GaussianBlur(p=1.0)

        global_transfo2_extra = transforms.Compose(
            [
                GaussianBlur(p=0.1),
                transforms.RandomSolarize(threshold=128, p=0.2),
            ]
        )

        local_transfo_extra = GaussianBlur(p=0.5)

        # normalization
        self.normalize = transforms.Compose(
            [
                transforms.ToTensor(),
                make_normalize_transform(),
            ]
        )

        self.global_transfo1 = transforms.Compose([color_jittering, global_transfo1_extra, self.normalize])
        self.global_transfo2 = transforms.Compose([color_jittering, global_transfo2_extra, self.normalize])
        self.local_transfo = transforms.Compose([color_jittering, local_transfo_extra, self.normalize])

    def __call__(self, image):
        output = {}
        image = (image+1)/2
        image = torchvision.transforms.functional.to_pil_image(image)
        # global crops:
        im1_base = self.geometric_augmentation_global(image)
        global_crop_1 = self.global_transfo1(im1_base)

        im2_base = self.geometric_augmentation_global(image)
        global_crop_2 = self.global_transfo2(im2_base)

        output["global_crops"] = [global_crop_1, global_crop_2]

        # global crops for teacher:
        output["global_crops_teacher"] = [global_crop_1, global_crop_2]

        # local crops:
        local_crops = [
            self.local_transfo(self.geometric_augmentation_local(image)) for _ in range(self.local_crops_number)
        ]
        output["local_crops"] = local_crops
        output["offsets"] = ()

        return output



In [ ]:
class DinoSubjectModel(nn.Module):
    def __init__(self, model_name='dinov2_vitb14'):
        super().__init__()
        self.model = torch.hub.load('facebookresearch/dinov2', model_name)
        self.model.eval()  # Set to eval mode
        self.preprocess = transforms.Compose([
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(224),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])
        
    def forward(self, x):
        x = (x+1)/2  # Scale from [-1, 1] to [0, 1]
        x = self.preprocess(x)
        return self.model(x)

    def decode(self, y):
        raise NotImplementedError("DINOv2 does not support decoding.")

In [ ]:
class ResNetSubjectModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torchvision.models.resnet50(
            weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
        )
        self.model.fc = nn.Identity()  # remove classifier
        self.model.eval()

        self.preprocess = torchvision.models.ResNet50_Weights.IMAGENET1K_V2.transforms()

    def forward(self, x):
        x = ((x+1)/2) 
        x = self.preprocess(x)
        feats = self.model(x)
        return feats

    def decode(self, y):
        raise NotImplementedError("ResNet does not support decoding.")

In [ ]:
from fff.evaluate.fid_old import InceptionV3Features

class InceptionSubjectModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = InceptionV3Features(device)
        self.model.eval()
        
    def forward(self, x):
        x = (x + 1) / 2
        # Resize to Inception input
        x = torch.nn.functional.interpolate(x, size=299, mode="bilinear", align_corners=False)
        return self.model(x)

    def decode(self, y):
        raise NotImplementedError("InceptionV3 does not support decoding.")

In [ ]:
class PretrainedClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = torchvision.models.convnext_large(weights=torchvision.models.ConvNeXt_Large_Weights.IMAGENET1K_V1).to(device)
        self.classifier = self.classifier.eval()
        self.transforms = torchvision.models.ConvNeXt_Large_Weights.IMAGENET1K_V1.transforms()
    
    def forward(self, x):
        x = self.transforms(x/2 + 0.5)
        return self.classifier(x)

# ImageNet DINO

In [ ]:
os.chdir(paths.output("imagenet"))

In [ ]:
combine_and_save_imagenet_samples(prefix_filter="sampled_imagenet_invariances_", save_name="sampled_imagenet_invariances_no_modification.pt")

In [ ]:
inv_dict = torch.load("combined_imagenet_invariances/v4/sampled_imagenet_invariances.pt")

In [ ]:
orig, inv, mask, nn_imgs = inv_dict["originals"], inv_dict["invariances"], inv_dict["masks"],  inv_dict["nn_imgs"]

In [ ]:
augmentor = DataAugmentationDINO(global_crops_scale=(0.32, 1.0), local_crops_scale=(0.05, 0.32), local_crops_number=8)
augmented = [augmentor(im)["global_crops"][0] for im in orig]

In [ ]:
subject_model = DinoSubjectModel().to(device)

with torch.no_grad():
    augmented_embeddings = subject_model(torch.stack(augmented, dim=0).to(device))
    assert ((inv_dict["original_embeddings"] - subject_model(orig))**2).mean() < 1.e-4

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "imagenet",
    "root": f"{paths.data()}",
    "resize_to": 256,
    "normalize": True,
}
_, val_ds, _ = load_dataset(**data_set_config)


nn_indices = find_nearest_neighbor_batched_mm(inv_dict["original_embeddings"], val_ds, subject_model)
nn_imgs = torch.stack([val_ds[ind][0] for ind in nn_indices], dim=0)

In [ ]:
inv_dict["nn_imgs"] = nn_imgs
with torch.no_grad():
    inv_dict["nn_embeddings"] = subject_model(nn_imgs.to(device))
inv_dict["nn_indices"] = nn_indices

In [ ]:
torch.save(inv_dict, "combined_imagenet_invariances/v4/sampled_imagenet_invariances.pt")

In [ ]:
with torch.no_grad():
    fiber_loss_invariances = ((inv_dict["original_embeddings"][:,None,...] - inv_dict["invariances_embeddings"])**2).sum(dim=-1)
    fiber_loss_nearest_neighbors = ((inv_dict["original_embeddings"] - inv_dict["nn_embeddings"])**2).sum(dim=-1)
    # fiber_loss_augmented = ((inv_dict["original_embeddings"] - augmented_embeddings)**2).sum(dim=-1)
    mean_fiber_loss_invariances = masked_mean(fiber_loss_invariances, inv_dict["masks"], 0)
    print(f"Fiber loss invariances: {mean_fiber_loss_invariances.mean()} +- {mean_fiber_loss_invariances.std()}")
    print(f"Fiber loss nearest neighbors: {fiber_loss_nearest_neighbors.mean()}")
    # print(f"Fiber loss augmented: {fiber_loss_augmented.mean()}")

In [ ]:
plt_nn = True
plt_aug = True
num_samples = 4
samples_per_image = 3
fontsize = 28

# plot_indices = torch.randperm(len(orig))[:num_samples]
# print(plot_indices)
plot_indices = [264, 199, 495, 33]
plots_per_row = 1 + samples_per_image + plt_nn + plt_aug
plt.figure(figsize=(5*plots_per_row, 5*num_samples))

for i, ind in enumerate(plot_indices): 
    plt.subplot(num_samples, plots_per_row, i*plots_per_row+1)
    plt.imshow(orig[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.title("Originals", fontsize=fontsize)

    if plt_nn:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2)
        plt.imshow(nn_imgs[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_nearest_neighbors[ind],), fontsize=fontsize)
        if i == 0:
            plt.title("Nearest Neighbor", fontsize=fontsize)
    if plt_aug:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2+plt_nn)
        plt.imshow(augmented[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_augmented[ind], ), fontsize=fontsize)
        if i == 0:
            plt.title("Augmented Sample", fontsize=fontsize)

    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_nn + plt_aug + j)
        fl_masked = fiber_loss_invariances[ind] + torch.where(mask[ind], 0.0, torch.inf).to(device)
        min_fl_ind = torch.argsort(fl_masked, dim=0, descending=False)[j]
        plt.imshow(inv[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_invariances[ind][min_fl_ind], ), fontsize=fontsize)
        if i == 0:
            plt.title("Invariant samples", fontsize=fontsize)
plt.savefig(f"{paths.output('figures')}/DinoSamples.jpeg", bbox_inches="tight")
plt.show()

In [ ]:
img_index = 9
num_inv_to_show = min(10, inv_dict["masks"][img_index].sum().item())
show_orig = True
plt.figure(figsize=(5*(num_inv_to_show+show_orig), 5))
save_samples = False


if show_orig:
    plt.subplot(1, num_inv_to_show+show_orig, 1)
    plt.imshow(inv_dict["originals"][img_index].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.axis("off")

current_sample_set = 0

for i in range(num_inv_to_show):
    while not inv_dict["masks"][img_index][current_sample_set]:
        current_sample_set += 1
        if current_sample_set >= inv_dict["masks"].shape[1]:
            raise(RuntimeError("Error computing the maximum number of samples to show"))
        
    plt.subplot(1, num_inv_to_show+show_orig, i+1+show_orig)
    plt.imshow(inv_dict["invariances"][img_index, current_sample_set].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.axis("off")
    current_sample_set += 1
plt.show()

if save_samples:
    plt.figure(figsize=(5, 5))
    plt.imshow(inv_dict["originals"][img_index].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.axis("off")
    plt.savefig("PaperSamples/ImageNetOriginal.pdf", bbox_inches="tight")
    
    current_sample_set = 0
    
    for i in range(num_inv_to_show):
        while not inv_dict["masks"][img_index][current_sample_set]:
            current_sample_set += 1
            if current_sample_set >= inv_dict["masks"].shape[1]:
                raise(RuntimeError("Error computing the maximum number of samples to show"))
        plt.imshow(inv_dict["invariances"][img_index, current_sample_set].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.axis("off")
        current_sample_set += 1
        plt.savefig(f"PaperSamples/ImageNetInvariantSample_{i+1}.pdf", bbox_inches="tight")


In [ ]:
pretrained_classifier = PretrainedClassifier()

@torch.no_grad()
def get_logits(pretrained_classifier, inputs, batch_size):
    dl = torch.utils.data.DataLoader(inputs, batch_size=batch_size, drop_last=False, shuffle=False)
    logits = []
    for batch in dl:
        logits.append(pretrained_classifier(batch))
    return torch.cat(logits, dim=0)

@torch.no_grad()
def get_labels(pretrained_classifier, inputs, batch_size):
    logits = get_logits(pretrained_classifier, inputs, batch_size)
    return torch.argmax(logits, dim=1)

@torch.no_grad()
def label_in_top_k(labels, logits, k=5):
    topk_indices = logits.topk(k, dim=1).indices

    labels = labels.unsqueeze(1)
    in_top_k = (topk_indices == labels).any(dim=1)
    return in_top_k

logits_orig = get_logits(pretrained_classifier, inv_dict["originals"], batch_size=512)
recomputed_labels = torch.argmax(logits_orig, dim=1)
classifer_top_k = label_in_top_k(inv_dict["labels"].to(device), logits_orig)

logits_inv = []
recomputed_in_top_k = []
gt_in_top_k = []

for i in tqdm(range(inv_dict["masks"].shape[1]), desc="Computing invariance logits"):
    mask, inv = inv_dict["masks"][:,i], inv_dict["invariances"][:,i]
    logits_inv.append(get_logits(pretrained_classifier, inv[mask].to(device), batch_size=512))
    recomputed_in_top_k.append(label_in_top_k(recomputed_labels[mask], logits_inv[-1]))
    gt_in_top_k.append(label_in_top_k(inv_dict["labels"][mask].to(device), logits_inv[-1]))


In [ ]:
top_k_acc = classifer_top_k.float().mean().item()
top_k_acc_inv = [sample_set.float().mean().item() for sample_set in gt_in_top_k]
top_k_recomputed_acc_inv = [sample_set.float().mean().item() for sample_set in recomputed_in_top_k]

print(f"Top K accuracy of classifier on originals: {np.mean(top_k_acc)*100:.2f}%")
print(f"Top K accuracy of classifier on invariances: {np.mean(top_k_acc_inv)*100:.2f}% +- {np.std(top_k_acc_inv)*100:.2f}")
print(f"Top K accuracy of classifier on invariances, using label classifier predict on gt: {np.mean(top_k_recomputed_acc_inv)*100:.2f}% +- {np.std(top_k_recomputed_acc_inv)*100:.2f}")

## Unconditional Samples

In [ ]:
combine_and_save_imagenet_samples(prefix_filter="sampled_imagenet_unconditional_", save_name="sampled_imagenet_unconditional.pt")

In [ ]:
unconditional_samples_dict = torch.load("sampled_imagenet_unconditional.pt")

In [ ]:
fids_unconditional = []
for i in range(unconditional_samples_dict["invariances"].shape[1]):
    inv, mask = unconditional_samples_dict["invariances"][:, i], unconditional_samples_dict["masks"][:, i]
    inv = torch.clamp(inv, -1, 1)
    fid = compute_fid(unconditional_samples_dict["originals"][:][mask].to(device), inv[mask].to(device))
    fids_unconditional.append(fid)

In [ ]:
print(f"FID of unconditional samples is {np.mean(fids_unconditional)} +- {np.std(fids_unconditional)}")

In [ ]:
inv, mask = unconditional_samples_dict["invariances"][:, 0], unconditional_samples_dict["masks"][:, 0]
orig = unconditional_samples_dict["originals"][mask]
num_samples = 4
mask_indices = torch.randperm(mask.sum())[:num_samples]
plot_indices = torch.where(torch.logical_and(torch.sum(mask.cumsum(dim=0).unsqueeze(1) - 1 == mask_indices.unsqueeze(0), dim=1), mask))[0]
plots_per_col = 2 
plt.figure(figsize=(5*num_samples, 5*plots_per_col))

for i, ind in enumerate(plot_indices):
    plt.subplot(plots_per_col, num_samples, i+1)
    plt.imshow(orig[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Originals", fontsize=14)
    plt.subplot(plots_per_col, num_samples, i+1+num_samples)
    plt.imshow(inv[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Invariant samples", fontsize=14)
plt.show()

## Sanity Check

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "imagenet",
    "root": f"{paths.data()}",
    "resize_to": 256,
    "normalize": True,
}

ds_np = np.load("precomputed_unconditional_samples.npz")["arr_0"]
ds_torch = (torch.from_numpy(ds_np).permute(0, 3, 1, 2)/255)*2 - 1
_, val_ds, _ = load_dataset(**data_set_config)
dataloader = torch.utils.data.DataLoader(val_ds, batch_size=10, shuffle=True)
ref_labels = iter(dataloader).__next__()[1]
print(ref_labels)

In [ ]:

num_samples = 4
plot_indices = torch.randperm(len(ds_torch))[:num_samples]
plots_per_col = 2 
plt.figure(figsize=(5*num_samples, 5*plots_per_col))

for i, ind in enumerate(plot_indices):
    plt.subplot(plots_per_col, num_samples, i+1)
    plt.imshow(ref_samples[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Originals", fontsize=14)
    plt.subplot(plots_per_col, num_samples, i+1+num_samples)
    plt.imshow(ds_torch[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Invariant samples", fontsize=14)
plt.show()

In [ ]:
fid = compute_fid(ds_torch, ref_samples)
print(fid)

## Different Fiber losses

### L1 Loss

In [ ]:
sequential_dict = load_and_combine_sequential_samples(prefix_filter="sampled_imagenet_invariances_c1")

In [ ]:
with torch.no_grad():
    fiber_loss_l2 = ((sequential_dict["original_embeddings"] - sequential_dict["invariances_embeddings"])**2).sum(dim=-1)
    fiber_loss_l1 = ((sequential_dict["original_embeddings"] - sequential_dict["invariances_embeddings"]).abs()).sum(dim=-1)
    fiber_loss_ce = torch.nn.functional.cross_entropy(sequential_dict["invariances_embeddings"], sequential_dict["original_embeddings"].softmax(dim=-1), reduction="none")
    fiber_loss_l2 = fiber_loss_l2[~fiber_loss_l2.isnan()]
    fiber_loss_l1 = fiber_loss_l1[~fiber_loss_l1.isnan()]
    fiber_loss_ce = fiber_loss_ce[~fiber_loss_ce.isnan()]
    print(f"Fiber loss l2: {fiber_loss_l2.mean()} +- {fiber_loss_l2.std()}")
    print(f"Fiber loss l1: {fiber_loss_l1.mean()} +- {fiber_loss_l1.std()}")
    print(f"Fiber loss CE: {fiber_loss_ce.mean()} +- {fiber_loss_ce.std()}")

### CE Loss

In [ ]:
sequential_dict = load_and_combine_sequential_samples(prefix_filter="sampled_imagenet_invariances_ce")

In [ ]:
with torch.no_grad():
    fiber_loss_l2 = ((sequential_dict["original_embeddings"] - sequential_dict["invariances_embeddings"])**2).sum(dim=-1)
    fiber_loss_l1 = ((sequential_dict["original_embeddings"] - sequential_dict["invariances_embeddings"]).abs()).sum(dim=-1)
    fiber_loss_ce = torch.nn.functional.cross_entropy(sequential_dict["invariances_embeddings"], sequential_dict["original_embeddings"].softmax(dim=-1), reduction="none")
    fiber_loss_l2 = fiber_loss_l2[~fiber_loss_l2.isnan()]
    fiber_loss_l1 = fiber_loss_l1[~fiber_loss_l1.isnan()]
    fiber_loss_ce = fiber_loss_ce[~fiber_loss_ce.isnan()]
    print(f"Fiber loss l2: {fiber_loss_l2.mean()} +- {fiber_loss_l2.std()}")
    print(f"Fiber loss l1: {fiber_loss_l1.mean()} +- {fiber_loss_l1.std()}")
    print(f"Fiber loss CE: {fiber_loss_ce.mean()} +- {fiber_loss_ce.std()}")

In [ ]:
plt_aug = True
num_samples = 5
fontsize = 24

plot_indices = torch.randperm(len(sequential_dict["originals"]))[:5]
plots_per_col = 3
plt.figure(figsize=(5*num_samples, 5*plots_per_col))


for i, ind in enumerate(plot_indices): 
    plt.subplot(plots_per_col, num_samples, i+1)
    plt.imshow(sequential_dict["originals"][ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Originals", fontsize=fontsize)

    plt.subplot(plots_per_col, num_samples, i + 1 + num_samples)
    plt.imshow(sequential_dict["invariances"][ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    fl_old = ((sequential_dict["original_embeddings"][ind] - sequential_dict["invariances_embeddings"][ind])**2).sum(dim=-1).item()
    plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fl_old, ), fontsize=fontsize)
    if i == 0:
        plt.ylabel("CE NDTM", fontsize=fontsize)
plt.savefig(f"{paths.output('figures')}/Stativ_vs_Moving_ImageNet.jpeg", bbox_inches="tight")
plt.show()

### L2 Loss

In [ ]:
inv_dict = torch.load("combined_imagenet_invariances/v4/sampled_imagenet_invariances.pt")

In [ ]:
with torch.no_grad():
    fiber_loss_l2 = ((inv_dict["original_embeddings"][:,None,...] - inv_dict["invariances_embeddings"])**2).sum(dim=-1)
    fiber_loss_l1 = ((inv_dict["original_embeddings"][:,None,...] - inv_dict["invariances_embeddings"]).abs()).sum(dim=-1)
    fiber_loss_ce = torch.zeros_like(fiber_loss_l2)
    for i in range(fiber_loss_ce.shape[1]):
        fiber_loss_ce[:,i] = torch.nn.functional.cross_entropy(inv_dict["invariances_embeddings"][:,i], inv_dict["original_embeddings"].softmax(dim=-1), reduction="none")
    
    print(f"Fiber loss l2: {masked_mean(fiber_loss_l2, inv_dict['masks'], 0).mean()} +- {masked_std(fiber_loss_l2, inv_dict['masks'], 0).mean()}")
    print(f"Fiber loss l1: {masked_mean(fiber_loss_l1, inv_dict['masks'], 0).mean()} +- {masked_std(fiber_loss_l1, inv_dict['masks'], 0).mean()}")
    print(f"Fiber loss CE: {masked_mean(fiber_loss_ce, inv_dict['masks'], 0).mean()} +- {masked_std(fiber_loss_ce, inv_dict['masks'], 0).mean()}")

### Nearest Neighbors

In [ ]:
subject_model = DinoSubjectModel().to(device)

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "imagenet",
    "root": f"{paths.data()}",
    "resize_to": 256,
    "normalize": True,
}
_, val_ds, _ = load_dataset(**data_set_config)


# nn_indices = find_nearest_neighbor_batched_mm(inv_dict["original_embeddings"], val_ds, subject_model)

In [ ]:
nn_indices_ce = find_nearest_neighbor_batched_mm(inv_dict["original_embeddings"], val_ds, subject_model, metric="cross_entropy")

In [ ]:
nn_indices_l1 = find_nearest_neighbor_batched_mm(inv_dict["original_embeddings"], val_ds, subject_model, metric="l1")

In [ ]:
inv_dict["nn_indices_ce"] = nn_indices_ce
inv_dict["nn_indices_l1"] = nn_indices_l1
torch.save(inv_dict, "combined_imagenet_invariances/v4/sampled_imagenet_invariances.pt")

In [ ]:
nn_imgs_ce = torch.stack([val_ds[ind][0] for ind in inv_dict["nn_indices_ce"]], dim=0)
with torch.no_grad():
    inv_dict["nn_embeddings_ce"] = subject_model(nn_imgs_ce.to(device))

In [ ]:
nn_imgs_l1 = torch.stack([val_ds[ind][0] for ind in inv_dict["nn_indices_l1"]], dim=0)
with torch.no_grad():
    inv_dict["nn_embeddings_l1"] = subject_model(nn_imgs_l1.to(device))

In [ ]:
with torch.no_grad():
    fiber_loss_l2 = ((inv_dict["original_embeddings"] - inv_dict["nn_embeddings"])**2).sum(dim=-1)
    fiber_loss_l1 = ((inv_dict["original_embeddings"] - inv_dict["nn_embeddings_l1"]).abs()).sum(dim=-1)
    fiber_loss_ce = torch.nn.functional.cross_entropy(inv_dict["nn_embeddings_ce"], inv_dict["original_embeddings"].softmax(dim=-1), reduction="none")

    print(f"Fiber loss l2: {fiber_loss_l2.mean()} +- {fiber_loss_l2.std()}")
    print(f"Fiber loss l1: {fiber_loss_l1.mean()} +- {fiber_loss_l1.std()}")
    print(f"Fiber loss CE: {fiber_loss_ce.mean()} +- {fiber_loss_ce.std()}")

## Old vs new NDTM

In [ ]:
inv_dict_old = torch.load("sampled_imagenet_invariances_14_31_15__26_01_2026_48983/chunk_0.pt")
inv_dict_new = torch.load("sampled_imagenet_invariances_15_10_59__26_01_2026_4593/chunk_0.pt")

In [ ]:
orig_old, inv_old = inv_dict_old["originals"], inv_dict_old["invariances"]
orig_new, inv_new = inv_dict_new["originals"], inv_dict_new["invariances"]

In [ ]:
plt_aug = True
num_samples = 5
fontsize = 24

plot_indices = [0, 9, 6, 7, 4]
plots_per_col = 3
plt.figure(figsize=(5*num_samples, 5*plots_per_col))


for i, ind in enumerate(plot_indices): 
    plt.subplot(plots_per_col, num_samples, i+1)
    plt.imshow(orig_old[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Originals", fontsize=fontsize)

    plt.subplot(plots_per_col, num_samples, i + 1 + num_samples)
    plt.imshow(inv_old[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    fl_old = ((inv_dict_old["original_embeddings"][ind] - inv_dict_old["invariances_embeddings"][ind])**2).sum(dim=-1).item()
    plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fl_old, ), fontsize=fontsize)
    if i == 0:
        plt.ylabel("Unmodified NDTM", fontsize=fontsize)

    plt.subplot(plots_per_col, num_samples, i + 1 + num_samples*2)
    plt.imshow(inv_new[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    fl_new = ((inv_dict_new["original_embeddings"][ind] - inv_dict_new["invariances_embeddings"][ind])**2).sum(dim=-1).item()
    plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fl_new, ), fontsize=fontsize)
    if i == 0:
        plt.ylabel("Modified NDTM", fontsize=fontsize)
plt.savefig(f"{paths.output('figures')}/Stativ_vs_Moving_ImageNet.jpeg", bbox_inches="tight")
plt.show()

# ImageNet Inception

In [ ]:
os.chdir(paths.output("imagenet"))

In [ ]:
combine_and_save_imagenet_samples(prefix_filter="sampled_inception_invariances_", save_name="sampled_inception_invariances.pt")

In [ ]:
inv_dict = torch.load("combined_imagenet_invariances/v5/sampled_inception_invariances.pt")

In [ ]:
orig, inv, mask = inv_dict["originals"], inv_dict["invariances"], inv_dict["masks"]
nn_imgs = inv_dict["nn_imgs"]

In [ ]:
augmentor = DataAugmentationDINO(global_crops_scale=(0.32, 1.0), local_crops_scale=(0.05, 0.32), local_crops_number=8)
augmented = [augmentor(im)["global_crops"][0] for im in orig]

In [ ]:
subject_model = InceptionSubjectModel().to(device)

with torch.no_grad():
    augmented_embeddings = subject_model(torch.stack(augmented, dim=0).to(device))
    assert ((inv_dict["original_embeddings"] - subject_model(orig))**2).mean() < 1.e-4

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "imagenet",
    "root": f"{paths.data()}",
    "resize_to": 256,
    "normalize": True,
}
_, val_ds, _ = load_dataset(**data_set_config)


nn_indices = find_nearest_neighbor_batched_mm(inv_dict["original_embeddings"], val_ds, subject_model)
nn_imgs = torch.stack([val_ds[ind][0] for ind in nn_indices], dim=0)

In [ ]:
inv_dict["nn_imgs"] = nn_imgs
with torch.no_grad():
    inv_dict["nn_embeddings"] = subject_model(nn_imgs.to(device))
inv_dict["nn_indices"] = nn_indices

In [ ]:
torch.save(inv_dict, "combined_imagenet_invariances/v5/sampled_inception_invariances.pt")

In [ ]:
with torch.no_grad():
    fiber_loss_invariances = ((inv_dict["original_embeddings"][:,None,...] - inv_dict["invariances_embeddings"])**2).sum(dim=-1)
    fiber_loss_nearest_neighbors = ((inv_dict["original_embeddings"] - inv_dict["nn_embeddings"])**2).sum(dim=-1)
    fiber_loss_augmented = ((inv_dict["original_embeddings"] - augmented_embeddings)**2).sum(dim=-1)
    mean_fiber_loss_invariances = masked_mean(fiber_loss_invariances, inv_dict["masks"], 0)
    print(f"Fiber loss invariances: {mean_fiber_loss_invariances.mean()} +- {mean_fiber_loss_invariances.std()}")
    print(f"Fiber loss nearest neighbors: {fiber_loss_nearest_neighbors.mean()}")
    print(f"Fiber loss augmented: {fiber_loss_augmented.mean()}")

In [ ]:
plt_nn = True
plt_aug = True
num_samples = 4
samples_per_image = 3
fontsize = 28

# plot_indices = torch.randperm(len(orig))[:num_samples]
# print(plot_indices)
plot_indices = [326, 690, 455, 794]
plots_per_row = 1 + samples_per_image + plt_nn + plt_aug
plt.figure(figsize=(5*plots_per_row, 5*num_samples))

for i, ind in enumerate(plot_indices): 
    plt.subplot(num_samples, plots_per_row, i*plots_per_row+1)
    plt.imshow(orig[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.title("Originals", fontsize=fontsize)

    if plt_nn:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2)
        plt.imshow(nn_imgs[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_nearest_neighbors[ind],), fontsize=fontsize)
        if i == 0:
            plt.title("Nearest Neighbor", fontsize=fontsize)
    if plt_aug:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2+plt_nn)
        plt.imshow(augmented[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_augmented[ind], ), fontsize=fontsize)
        if i == 0:
            plt.title("Augmented Sample", fontsize=fontsize)

    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_nn + plt_aug + j)
        fl_masked = fiber_loss_invariances[ind] + torch.where(mask[ind], 0.0, torch.inf).to(device)
        min_fl_ind = torch.argsort(fl_masked, dim=0, descending=False)[j]
        plt.imshow(inv[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fiber_loss_invariances[ind][min_fl_ind], ), fontsize=fontsize)
        if i == 0:
            plt.title("Invariant samples", fontsize=fontsize)
plt.savefig(f"{paths.output('figures')}/InceptionSamples.jpeg", bbox_inches="tight")
plt.show()

In [ ]:
pretrained_classifier = PretrainedClassifier()

@torch.no_grad()
def get_logits(pretrained_classifier, inputs, batch_size):
    dl = torch.utils.data.DataLoader(inputs, batch_size=batch_size, drop_last=False, shuffle=False)
    logits = []
    for batch in dl:
        logits.append(pretrained_classifier(batch))
    return torch.cat(logits, dim=0)

@torch.no_grad()
def get_labels(pretrained_classifier, inputs, batch_size):
    logits = get_logits(pretrained_classifier, inputs, batch_size)
    return torch.argmax(logits, dim=1)

@torch.no_grad()
def label_in_top_k(labels, logits, k=5):
    topk_indices = logits.topk(k, dim=1).indices

    labels = labels.unsqueeze(1)
    in_top_k = (topk_indices == labels).any(dim=1)
    return in_top_k

logits_orig = get_logits(pretrained_classifier, inv_dict["originals"], batch_size=512)
recomputed_labels = torch.argmax(logits_orig, dim=1)
classifer_top_k = label_in_top_k(inv_dict["labels"].to(device), logits_orig)

logits_inv = []
recomputed_in_top_k = []
gt_in_top_k = []

for i in tqdm(range(inv_dict["masks"].shape[1]), desc="Computing invariance logits"):
    mask, inv = inv_dict["masks"][:,i], inv_dict["invariances"][:,i]
    logits_inv.append(get_logits(pretrained_classifier, inv[mask].to(device), batch_size=512))
    recomputed_in_top_k.append(label_in_top_k(recomputed_labels[mask], logits_inv[-1]))
    gt_in_top_k.append(label_in_top_k(inv_dict["labels"][mask].to(device), logits_inv[-1]))


In [ ]:
top_k_acc = classifer_top_k.float().mean().item()
top_k_acc_inv = [sample_set.float().mean().item() for sample_set in gt_in_top_k]
top_k_recomputed_acc_inv = [sample_set.float().mean().item() for sample_set in recomputed_in_top_k]

print(f"Top K accuracy of classifier on originals: {np.mean(top_k_acc)*100:.2f}%")
print(f"Top K accuracy of classifier on invariances: {np.mean(top_k_acc_inv)*100:.2f}% +- {np.std(top_k_acc_inv)*100:.2f}")
print(f"Top K accuracy of classifier on invariances, using label classifier predict on gt: {np.mean(top_k_recomputed_acc_inv)*100:.2f}% +- {np.std(top_k_recomputed_acc_inv)*100:.2f}")

# Cue Conflict

In [ ]:
def combine_and_save_cue_conflict_samples(prefix_filter="sampled_cue_conflict_invariances_", save_name="sampled_cue_conflict_invariances.pt"):
    combined_dict = {
        "invariances_dino": [],
        "invariances_resnet": [],
        "originals": [],
        "shape_labels": [],
        "texture_labels": [],
        "invariances_dino_embeddings": [],
        "invariances_resnet_embeddings": [],
        "original_dino_embeddings": [],
        "original_resnet_embeddings": [],
        "invariances_dino_cross_embeddings": [],
        "invariances_resnet_cross_embeddings": [],
    }

    for foldername in tqdm(os.listdir(".")):
        if foldername.startswith(prefix_filter):
            results_dict = {
                "invariances_dino": [],
                "invariances_resnet": [],
                "originals": [],
                "shape_labels": [],
                "texture_labels": [],
                "invariances_dino_embeddings": [],
                "invariances_resnet_embeddings": [],
                "original_dino_embeddings": [],
                "original_resnet_embeddings": [],
                "invariances_dino_cross_embeddings": [],
                "invariances_resnet_cross_embeddings": [],
            }
            chunk_id = 0
            while os.path.exists(os.path.join(foldername, f"chunk_{chunk_id}.pt")):
                chunk_dict = torch.load(os.path.join(foldername, f"chunk_{chunk_id}.pt"))
                for key in results_dict.keys():
                    results_dict[key].append(chunk_dict[key])
                chunk_id += 1
            for key in results_dict.keys():
                combined_dict[key].append(torch.cat(results_dict[key], dim=0))
    
    # Check consistency:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        assert torch.all(((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=-1) < 1.e-4), "Dataset order does not agree"
    
    max_len, ind_max = 0, None
    for i, orig in enumerate(combined_dict["originals"]):
        if len(orig) > max_len:
            ind_max = i
            max_len = len(orig)
    
    combined_dict["originals"] = combined_dict["originals"][ind_max]
    combined_dict["original_dino_embeddings"] = combined_dict["original_dino_embeddings"][ind_max]
    combined_dict["original_resnet_embeddings"] = combined_dict["original_resnet_embeddings"][ind_max]
    combined_dict["shape_labels"] = combined_dict["shape_labels"][ind_max].to(device)
    combined_dict["texture_labels"] = combined_dict["texture_labels"][ind_max].to(device)
    
    assert len(combined_dict["originals"]) == len(combined_dict["original_dino_embeddings"]) == len(combined_dict["original_resnet_embeddings"])
    assert len(combined_dict["originals"]) == len(combined_dict["shape_labels"]) == len(combined_dict["texture_labels"])
    
    combined_dict["masks"] = []
    for i, (inv_dino, 
            inv_dino_embeddings, 
            inv_dino_cross_embeddings, 
            inv_resnet, 
            inv_resnet_embeddings, 
            inv_resnet_cross_embeddings) in enumerate(zip(combined_dict["invariances_dino"], 
                                                           combined_dict["invariances_dino_embeddings"],
                                                           combined_dict["invariances_dino_cross_embeddings"],
                                                           combined_dict["invariances_resnet"],
                                                           combined_dict["invariances_resnet_embeddings"],
                                                           combined_dict["invariances_resnet_cross_embeddings"])):
        
        assert len(inv_dino) == len(inv_dino_embeddings) == len(inv_resnet) == len(inv_resnet_embeddings) == len(inv_dino_cross_embeddings) == len(inv_resnet_cross_embeddings)
        mask = torch.cat((torch.ones(len(inv_dino), device=inv_dino.device, dtype=bool), 
                          torch.zeros(max_len - len(inv_dino), device=inv_dino.device, dtype=bool)), dim=0)
        inv_dino = torch.cat((inv_dino, torch.zeros(max_len - len(inv_dino), *inv_dino.shape[1:], device=inv_dino.device, dtype=inv_dino.dtype)), dim=0)
        inv_dino_embeddings = torch.cat((inv_dino_embeddings, torch.zeros(max_len - len(inv_dino_embeddings), *inv_dino_embeddings.shape[1:], device=inv_dino_embeddings.device, dtype=inv_dino_embeddings.dtype)), dim=0)
        inv_dino_cross_embeddings = torch.cat((inv_dino_cross_embeddings, torch.zeros(max_len - len(inv_dino_cross_embeddings), *inv_dino_cross_embeddings.shape[1:], device=inv_dino_cross_embeddings.device, dtype=inv_dino_cross_embeddings.dtype)), dim=0)
        inv_resnet = torch.cat((inv_resnet, torch.zeros(max_len - len(inv_resnet), *inv_resnet.shape[1:], device=inv_resnet.device, dtype=inv_resnet.dtype)), dim=0)
        inv_resnet_embeddings = torch.cat((inv_resnet_embeddings, torch.zeros(max_len - len(inv_resnet_embeddings), *inv_resnet_embeddings.shape[1:], device=inv_resnet_embeddings.device, dtype=inv_resnet_embeddings.dtype)), dim=0)
        inv_resnet_cross_embeddings = torch.cat((inv_resnet_cross_embeddings, torch.zeros(max_len - len(inv_resnet_cross_embeddings), *inv_resnet_cross_embeddings.shape[1:], device=inv_resnet_cross_embeddings.device, dtype=inv_resnet_cross_embeddings.dtype)), dim=0)
        combined_dict["invariances_dino"][i] = inv_dino
        combined_dict["invariances_dino_embeddings"][i] = inv_dino_embeddings
        combined_dict["invariances_dino_cross_embeddings"][i] = inv_dino_cross_embeddings
        combined_dict["invariances_resnet"][i] = inv_resnet
        combined_dict["invariances_resnet_embeddings"][i] = inv_resnet_embeddings
        combined_dict["invariances_resnet_cross_embeddings"][i] = inv_resnet_cross_embeddings
        combined_dict["masks"].append(mask)
    
    combined_dict["invariances_dino"] = torch.stack(combined_dict["invariances_dino"], dim=1)
    combined_dict["invariances_dino_embeddings"] = torch.stack(combined_dict["invariances_dino_embeddings"], dim=1)
    combined_dict["invariances_dino_cross_embeddings"] = torch.stack(combined_dict["invariances_dino_cross_embeddings"], dim=1)
    combined_dict["invariances_resnet"] = torch.stack(combined_dict["invariances_resnet"], dim=1)
    combined_dict["invariances_resnet_embeddings"] = torch.stack(combined_dict["invariances_resnet_embeddings"], dim=1)
    combined_dict["invariances_resnet_cross_embeddings"] = torch.stack(combined_dict["invariances_resnet_cross_embeddings"], dim=1)
    combined_dict["masks"] = torch.stack(combined_dict["masks"], dim=1)
    
    torch.save(combined_dict, save_name)

In [ ]:
os.chdir(paths.output("imagenet"))

In [ ]:
combine_and_save_cue_conflict_samples(prefix_filter="sampled_cue_conflict_invariances_", save_name="sampled_cue_conflict_invariances.pt")

In [ ]:
inv_dict = torch.load("combined_imagenet_invariances/v5/sampled_cue_conflict_invariances.pt")

In [ ]:
orig, inv_dino, inv_resnet = inv_dict["originals"], inv_dict["invariances_dino"], inv_dict["invariances_resnet"]
mask = inv_dict["masks"].to(device)
nn_indices_dino = inv_dict["nn_indices_dino"]
nn_imgs_dino = inv_dict["nn_imgs_dino"]
nn_indices_resnet = inv_dict["nn_indices_resnet"]
nn_imgs_resnet = inv_dict["nn_imgs_resnet"]
shape_labels, texture_labels = inv_dict["shape_labels"], inv_dict["texture_labels"]

In [ ]:
mask.shape

In [ ]:
augmentor = DataAugmentationDINO(global_crops_scale=(0.32, 1.0), local_crops_scale=(0.05, 0.32), local_crops_number=8)
augmented = [augmentor(im)["global_crops"][0] for im in orig]

In [ ]:
subject_model_dino = DinoSubjectModel().to(device)
subject_model_resnet = ResNetSubjectModel().to(device)

with torch.no_grad():
    augmented_dino_embeddings = subject_model_dino(torch.stack(augmented, dim=0).to(device))
    augmented_resnet_embeddings = subject_model_resnet(torch.stack(augmented, dim=0).to(device))
    assert ((inv_dict["original_dino_embeddings"] - subject_model_dino(orig))**2).mean() < 1.e-4
    assert ((inv_dict["original_resnet_embeddings"] - subject_model_resnet(orig))**2).mean() < 1.e-4

In [ ]:
from fff.data import load_dataset

data_set_config = {
    "name": "cue_conflict",
    "root": f"{paths.data('texture-vs-shape')}/stimuli/style-transfer-preprocessed-512",
    "resize_to": 256,
    "normalize": True,
}
_, val_ds, _ = load_dataset(**data_set_config)

In [ ]:
nn_indices_dino = find_nearest_neighbor_batched_mm(inv_dict["original_dino_embeddings"], val_ds, subject_model_dino, batch_size=256)
nn_imgs_dino = torch.stack([val_ds[ind][0] for ind in nn_indices_dino], dim=0)

In [ ]:
nn_indices_resnet = find_nearest_neighbor_batched_mm(inv_dict["original_resnet_embeddings"], val_ds, subject_model_resnet, batch_size=256)
nn_imgs_resnet = torch.stack([val_ds[ind][0] for ind in nn_indices_resnet], dim=0)

In [ ]:
inv_dict["nn_imgs_resnet"] = nn_imgs_resnet
inv_dict["nn_imgs_dino"] = nn_imgs_dino
with torch.no_grad():
    inv_dict["nn_resnet_embeddings"] = subject_model_resnet(nn_imgs_resnet.to(device))
    inv_dict["nn_dino_embeddings"] = subject_model_dino(nn_imgs_dino.to(device))
    inv_dict["nn_resnet_cross_embeddings"] = subject_model_dino(nn_imgs_resnet.to(device))
    inv_dict["nn_dino_cross_embeddings"] = subject_model_resnet(nn_imgs_dino.to(device))
inv_dict["nn_indices_resnet"] = nn_indices_resnet
inv_dict["nn_indices_dino"] = nn_indices_dino

In [ ]:
torch.save(inv_dict, "combined_imagenet_invariances/v5/sampled_cue_conflict_invariances.pt")

In [ ]:
with torch.no_grad():
    fiber_loss_dino = ((inv_dict["original_dino_embeddings"][:,None,...] - inv_dict["invariances_dino_embeddings"])**2).sum(dim=-1)
    fiber_loss_resnet = ((inv_dict["original_resnet_embeddings"][:,None,...] - inv_dict["invariances_resnet_embeddings"])**2).sum(dim=-1)
    fiber_loss_dino_nearest_neighbors = ((inv_dict["original_dino_embeddings"] - inv_dict["nn_dino_embeddings"])**2).sum(dim=-1)
    fiber_loss_resnet_nearest_neighbors = ((inv_dict["original_resnet_embeddings"] - inv_dict["nn_resnet_embeddings"])**2).sum(dim=-1)
    fiber_loss_dino_augmented = ((inv_dict["original_dino_embeddings"] - augmented_dino_embeddings)**2).sum(dim=-1)
    fiber_loss_resnet_augmented = ((inv_dict["original_resnet_embeddings"] - augmented_resnet_embeddings)**2).sum(dim=-1)
    mean_fiber_loss_dino_invariances = masked_mean(fiber_loss_dino, inv_dict["masks"], 0)
    mean_fiber_loss_resnet_invariances = masked_mean(fiber_loss_resnet, inv_dict["masks"], 0)
    print(f"Fiber loss DINO invariances: {mean_fiber_loss_dino_invariances.mean()} +- {mean_fiber_loss_dino_invariances.std()}")
    print(f"Fiber loss ResNet invariances: {mean_fiber_loss_resnet_invariances.mean()} +- {mean_fiber_loss_resnet_invariances.std()}")
    print(f"Fiber loss DINO nearest neighbors: {fiber_loss_dino_nearest_neighbors.mean()}")
    print(f"Fiber loss ResNet nearest neighbors: {fiber_loss_resnet_nearest_neighbors.mean()}")
    print(f"Fiber loss DINO augmented: {fiber_loss_dino_augmented.mean()}")
    print(f"Fiber loss ResNet augmented: {fiber_loss_resnet_augmented.mean()}")

## Pretrained Classifier

In [ ]:
import sys
sys.path.append(f"{paths.data('texture-vs-shape')}/code")
# sys.path.append(f"{paths.data('texture-vs-shape')}/code/helper")
from probabilities_to_decision import ImageNetProbabilitiesTo16ClassesMapping
from helper.human_categories import get_human_object_recognition_categories
import torch
import numpy as np
from typing import List, Union

class TorchImageNetProbabilitiesTo16Classes:
    def __init__(self, aggregation_function=np.mean, apply_softmax=False):
        """
        Parameters
        ----------
        aggregation_function : callable
            Function used to aggregate probabilities over ImageNet indices
            (default: np.mean, as in the original code).
        apply_softmax : bool
            If True, applies softmax to inputs before mapping.
            Set this to True if you pass logits instead of probabilities.
        """
        self.mapper = ImageNetProbabilitiesTo16ClassesMapping(
            aggregation_function=aggregation_function
        )
        self.apply_softmax = apply_softmax

    @torch.no_grad()
    def __call__(
        self,
        x: Union[torch.Tensor, np.ndarray]
    ) -> Union[str, List[str]]:
        """
        Parameters
        ----------
        x : torch.Tensor or np.ndarray
            Shape (1000,) or (B, 1000)

        Returns
        -------
        decision : str or List[str]
            One category if input is 1D, else a list of categories.
        """

        if isinstance(x, torch.Tensor):
            if self.apply_softmax:
                x = torch.softmax(x, dim=-1)

            x = x.detach().cpu().numpy()

        if x.ndim == 1:
            return self.mapper.probabilities_to_decision(x)

        elif x.ndim == 2:
            return [
                self.mapper.probabilities_to_decision(x[i])
                for i in range(x.shape[0])
            ]

        else:
            raise ValueError(f"Expected shape (1000,) or (B, 1000), got {x.shape}")


decision_fc = TorchImageNetProbabilitiesTo16Classes()
categories = get_human_object_recognition_categories()

In [ ]:
pretrained_classifier = PretrainedClassifier()

In [ ]:
def get_acc_from_categories(cat_list_1, cat_list_2):
    assert len(cat_list_1) == len(cat_list_2)
    agreement = [cat1 == cat2 for cat1, cat2 in zip(cat_list_1, cat_list_2)]
    agreement = torch.tensor(agreement, dtype=bool)
    return agreement.float().mean().item()

def get_acc_combined(cat_list_1, target_list_1, target_list_2):
    assert len(cat_list_1) == len(target_list_1) == len(target_list_2)
    agreement1 = [cat1 == cat2 for cat1, cat2 in zip(cat_list_1, target_list_1)]
    agreement2 = [cat1 == cat2 for cat1, cat2 in zip(cat_list_1, target_list_2)]
    agreement1 = torch.tensor(agreement1, dtype=bool)
    agreement2 = torch.tensor(agreement2, dtype=bool)
    return torch.logical_or(agreement1, agreement2).float().mean().item()



with torch.no_grad():
    logits_orig = pretrained_classifier(orig)
    logits_aug = pretrained_classifier(torch.stack(augmented, dim=0).to(device))

cat_shape = [categories[ind] for ind in inv_dict["shape_labels"]]
cat_texture = [categories[ind] for ind in inv_dict["texture_labels"]]
same_cat = torch.tensor([shape == texture for shape, texture in zip(cat_shape, cat_texture)], dtype=bool)

cat_orig = decision_fc(logits_orig.softmax(dim=1).cpu().detach().numpy())
cat_aug = decision_fc(logits_aug.softmax(dim=1).cpu().detach().numpy())

acc_shape_orig = get_acc_from_categories(cat_shape, cat_orig)
acc_texture_orig = get_acc_from_categories(cat_texture, cat_orig)
acc_both_orig = get_acc_combined(cat_orig, cat_shape, cat_texture)



In [ ]:
acc_shape_dino = []
acc_texture_dino = []
acc_both_dino = []

acc_shape_resnet = []
acc_texture_resnet = []
acc_both_resnet = []

cat_combined_dino = []
cat_combined_resnet = []

num_data_per_set = []

for sample_set in range(inv_dino.shape[1]):
    cat_sample_mask = torch.logical_and(mask[:,sample_set].cpu(), ~same_cat)
    with torch.no_grad():
        logits_dino = pretrained_classifier(inv_dino[cat_sample_mask, sample_set].to(device))
        logits_resnet = pretrained_classifier(inv_resnet[cat_sample_mask, sample_set].to(device))
    cat_dino = decision_fc(logits_dino.softmax(dim=1).cpu().detach().numpy())
    cat_resnet = decision_fc(logits_resnet.softmax(dim=1).cpu().detach().numpy())
    cat_combined_dino.append(cat_dino)
    cat_combined_resnet.append(cat_resnet)
    
    cat_shape_masked = [cat for i, cat in enumerate(cat_shape) if cat_sample_mask[i]]
    cat_texture_masked = [cat for i, cat in enumerate(cat_texture) if cat_sample_mask[i]]
    acc_shape_dino.append(get_acc_from_categories(cat_shape_masked, cat_dino))
    acc_texture_dino.append(get_acc_from_categories(cat_texture_masked, cat_dino))
    acc_both_dino.append(get_acc_combined(cat_dino, cat_shape_masked, cat_texture_masked))

    acc_shape_resnet.append(get_acc_from_categories(cat_shape_masked, cat_resnet))
    acc_texture_resnet.append(get_acc_from_categories(cat_texture_masked, cat_resnet))
    acc_both_resnet.append(get_acc_combined(cat_resnet, cat_shape_masked, cat_texture_masked))
    num_data_per_set.append(cat_sample_mask.sum())

In [ ]:
print([categories[ind] for ind in inv_dict["shape_labels"][:5]])
print([categories[ind] for ind in inv_dict["texture_labels"][:5]])
print(decision_fc(logits_orig[:5].softmax(dim=1).cpu().detach().numpy()))
print(decision_fc(logits_dino[:5].softmax(dim=1).cpu().detach().numpy()))
print(decision_fc(logits_resnet[:5].softmax(dim=1).cpu().detach().numpy()))

In [ ]:
print(f"Shape accuracy on originals: {acc_shape_orig*100:.1f}%")
print(f"Texture accuracy on originals: {acc_texture_orig*100:.1f}%")
print(f"Combined accuracy on originals: {acc_both_orig*100:.1f}%")

print(f"Shape accuracy on dino: {np.average(acc_shape_dino, weights=num_data_per_set)*100:.1f} +- {np.std(acc_shape_dino)*100:.1f}%")
print(f"Texture accuracy on dino: {np.average(acc_texture_dino, weights=num_data_per_set)*100:.1f} +- {np.std(acc_texture_dino)*100:.1f}%")
print(f"Combined accuracy on dino: {np.average(acc_both_dino, weights=num_data_per_set)*100:.1f} +- {np.std(acc_both_dino)*100:.1f}%")

print(f"Shape accuracy on resnet: {np.average(acc_shape_resnet, weights=num_data_per_set)*100:.1f} +- {np.std(acc_shape_resnet)*100:.1f}%")
print(f"Texture accuracy on resnet: {np.average(acc_texture_resnet, weights=num_data_per_set)*100:.1f} +- {np.std(acc_texture_resnet)*100:.1f}%")
print(f"Combined accuracy on resnet: {np.average(acc_both_resnet, weights=num_data_per_set)*100:.1f} +- {np.std(acc_both_resnet)*100:.1f}%")

In [ ]:
num_data_per_set

In [ ]:
rel_acc_shape_orig = acc_shape_orig/acc_both_orig
rel_acc_texture_orig = acc_texture_orig/acc_both_orig
rel_acc_shape_dino = [sh/bo for sh, bo in zip(acc_shape_dino, acc_both_dino)]
rel_acc_texture_dino = [sh/bo for sh, bo in zip(acc_texture_dino, acc_both_dino)]
rel_acc_shape_resnet = [sh/bo for sh, bo in zip(acc_shape_resnet, acc_both_resnet)]
rel_acc_texture_resnet = [sh/bo for sh, bo in zip(acc_texture_resnet, acc_both_resnet)]

# Means
shape_means = [
    rel_acc_shape_orig * 100,
    np.mean(rel_acc_shape_dino) * 100,
    np.mean(rel_acc_shape_resnet) * 100,
]

texture_means = [
    rel_acc_texture_orig * 100,
    np.mean(rel_acc_texture_dino) * 100,
    np.mean(rel_acc_texture_resnet) * 100,
]

# Std devs (0 for originals)
shape_stds = [
    0.0,
    np.std(rel_acc_shape_dino) * 100,
    np.std(rel_acc_shape_resnet) * 100,
]

texture_stds = [
    0.0,
    np.std(rel_acc_texture_dino) * 100,
    np.std(rel_acc_texture_resnet) * 100,
]

labels = ["Originals", "DINO Invariant Samples", "ResNet Invariant Samples"]
x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(7, 4), dpi=200)

plt.bar(
    x - width / 2,
    shape_means,
    width,
    yerr=shape_stds,
    label="Shape as GT",
    capsize=4,
)

plt.bar(
    x + width / 2,
    texture_means,
    width,
    yerr=texture_stds,
    label="Texture as GT",
    capsize=4,
)

plt.xticks(x, labels)
plt.ylabel("Label preference (%)")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.savefig(f"{paths.output('figures')}/CueConflictLabelPreference.pdf", bbox_inches="tight")
plt.show()


In [ ]:
plt_aug = True
num_samples = 5
samples_per_image = 2
fontsize = 24

combined_mask = ~same_cat
for sample_set in range(inv_dino.shape[1]):
    combined_mask = torch.logical_and(mask[:,sample_set].cpu(), combined_mask)

orig_masked = orig[combined_mask]
inv_dino_masked = inv_dino[combined_mask]
inv_resnet_masked = inv_resnet[combined_mask]
augmented_masked = torch.stack(augmented, dim=0)[combined_mask]
fl_dino_masked = fiber_loss_dino[combined_mask]
fl_resnet_masked = fiber_loss_resnet[combined_mask]
cat_shape_masked = [cat for i, cat in enumerate(cat_shape) if combined_mask[i]]
cat_texture_masked = [cat for i, cat in enumerate(cat_texture) if combined_mask[i]]
cat_orig_masked = [cat for i, cat in enumerate(cat_orig) if combined_mask[i]]
cat_aug_masked = [cat for i, cat in enumerate(cat_aug) if combined_mask[i]]


#plot_indices = torch.randperm(combined_mask.sum())[:num_samples]
#print(plot_indices)
plot_indices = [160, 224, 180, 114]
plots_per_row = 1 + samples_per_image*2 + plt_aug
plt.figure(figsize=(5*plots_per_row, 5*num_samples))


for i, ind in enumerate(plot_indices): 
    plt.subplot(num_samples, plots_per_row, i*plots_per_row+1)
    plt.imshow(orig_masked[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    plt.ylabel(f"Shape: {cat_shape_masked[ind]}\nTexture: {cat_texture_masked[ind]}", fontsize=fontsize)
    plt.xlabel(f"ConvNeXt: {cat_orig_masked[ind]}", fontsize=fontsize)

    if i == 0:
        plt.title("Originals", fontsize=fontsize)

    if plt_aug:
        plt.subplot(num_samples, plots_per_row, i*plots_per_row+2)
        plt.imshow(augmented_masked[ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(f"ConvNeXt: {cat_aug_masked[ind]}", fontsize=fontsize)
        if i == 0:
            plt.title("Augmented", fontsize=fontsize)

    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_aug + j)
        min_fl_ind = torch.argsort(fl_dino_masked[ind], dim=0, descending=False)[j]
        plt.imshow(inv_dino_masked[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(f"ConvNeXt: {cat_combined_dino[min_fl_ind][ind]}", fontsize=fontsize)
        if i == 0:
            plt.title("DINO Sample", fontsize=fontsize)

    for j in range(samples_per_image):
        plt.subplot(num_samples, plots_per_row, i*plots_per_row + 2 + plt_aug + j + samples_per_image)
        min_fl_ind = torch.argsort(fl_resnet_masked[ind], dim=0, descending=False)[j]
        plt.imshow(inv_resnet_masked[ind, min_fl_ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel(f"ConvNeXt: {cat_combined_resnet[min_fl_ind][ind]}", fontsize=fontsize)
        if i == 0:
            plt.title("ResNet Sample", fontsize=fontsize)
            
plt.savefig(f"{paths.output('figures')}/CueConflictSamples.jpeg", bbox_inches="tight")
plt.show()

# FID computation

## Dino

In [ ]:
def load_and_combine_sequential_samples(prefix_filter="sampled_imagenet_invariances_", root=f"{paths.output('imagenet')}"):
    combined_dict = {
        "invariances": [],
        "originals": [],
        "labels": [],
        "invariances_embeddings": [],
        "original_embeddings": [],
    }
    
    for foldername in tqdm(os.listdir(root)):
        if foldername.startswith(prefix_filter):
            results_dict = {
                            "invariances": [],
                            "originals": [],
                            "labels": [],
                            "invariances_embeddings": [],
                            "original_embeddings": [],
                           }
            for fname in os.listdir(os.path.join(root, foldername)):
                if "chunk_" in fname:
                    chunk_dict = torch.load(os.path.join(root, foldername, fname))
                    for key in results_dict.keys():
                        results_dict[key].append(chunk_dict[key])
            if len(results_dict["invariances"]) == 0:
                continue
            for key in results_dict.keys():
                combined_dict[key].append(torch.cat(results_dict[key], dim=0))
    # Check difference:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        duplicates = min_len - (((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=(1, 2, 3)) > 1.e-4).sum()
        assert duplicates == 0, f"Detected {duplicates//2} duplications"
        
    for key in combined_dict.keys():
        combined_dict[key] = torch.cat(combined_dict[key], dim=0)

    return combined_dict

In [ ]:
sequential_dict = load_and_combine_sequential_samples(prefix_filter="sampled_imagenet_invariances_")

In [ ]:
fid = compute_fid(sequential_dict["originals"], sequential_dict["invariances"])

In [ ]:
fid

In [ ]:
compute_fid_alternative(sequential_dict["originals"].to(device), sequential_dict["invariances"].to(device), batch_size=512)

## Inception Net

In [ ]:
def load_and_combine_sequential_samples(prefix_filter="sampled_imagenet_invariances_", root=f"{paths.output('imagenet')}"):
    combined_dict = {
        "invariances": [],
        "originals": [],
        "labels": [],
        "invariances_embeddings": [],
        "original_embeddings": [],
    }
    
    for foldername in tqdm(os.listdir(root)):
        if foldername.startswith(prefix_filter):
            results_dict = {
                            "invariances": [],
                            "originals": [],
                            "labels": [],
                            "invariances_embeddings": [],
                            "original_embeddings": [],
                           }
            for fname in os.listdir(os.path.join(root, foldername)):
                if "chunk_" in fname:
                    chunk_dict = torch.load(os.path.join(root, foldername, fname))
                    for key in results_dict.keys():
                        results_dict[key].append(chunk_dict[key])
            for key in results_dict.keys():
                combined_dict[key].append(torch.cat(results_dict[key], dim=0))
    # Check difference:
    for last_orig, next_orig in zip(combined_dict["originals"][:-1], combined_dict["originals"][1:]):
        min_len = min(len(last_orig), len(next_orig))
        assert torch.all(((last_orig[:min_len] - next_orig[:min_len])**2).sum(dim=(1, 2, 3)) > 1.e-4), "Detected duplications"
        
    for key in combined_dict.keys():
        combined_dict[key] = torch.cat(combined_dict[key], dim=0)

    return combined_dict

In [ ]:
sequential_dict = load_and_combine_sequential_samples(prefix_filter="sampled_inception_invariances_")

In [ ]:
fid = compute_fid(sequential_dict["originals"], sequential_dict["invariances"])

In [ ]:
fid

In [ ]:
compute_fid_alternative(sequential_dict["originals"].to(device), sequential_dict["invariances"].to(device), batch_size=512)

# Unmodified NDTM Fiber loss

In [ ]:
with torch.no_grad():
    fiber_loss_invariances = ((sequential_dict["original_embeddings"] - sequential_dict["invariances_embeddings"])**2).sum(dim=-1)
    print(f"Fiber loss invariances: {fiber_loss_invariances.mean()} +- {fiber_loss_invariances.std()}")

In [ ]:
num_samples = 5
fontsize = 24

plot_indices = torch.randperm(10000)[:5]
plots_per_col = 2
plt.figure(figsize=(5*num_samples, 5*plots_per_col))


for i, ind in enumerate(plot_indices): 
    plt.subplot(plots_per_col, num_samples, i+1)
    plt.imshow(sequential_dict["originals"][ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    if i == 0:
        plt.ylabel("Originals", fontsize=fontsize)

    plt.subplot(plots_per_col, num_samples, i + 1 + num_samples)
    plt.imshow(sequential_dict["invariances"][ind].permute(1, 2, 0).cpu().detach().numpy()/2 + 0.5)
    plt.xticks([])
    plt.yticks([])
    fl_old = ((sequential_dict["original_embeddings"][ind] - sequential_dict["invariances_embeddings"][ind])**2).sum(dim=-1).item()
    plt.xlabel(r"$\mathcal{L}_\text{fiber} = %.1f$" % (fl_old, ), fontsize=fontsize)
    if i == 0:
        plt.ylabel("Unmodified NDTM", fontsize=fontsize)


In [ ]:
len(sequential_dict["originals"])

In [ ]:
!ls